# 20260720*EDA_2021*전국\_시도분리정제

- 원본 작성자: 이정연
- 수정자: 지준구
- #5 이슈 참고
- 공통 함수는 일단 따로 안만들고, 이 노트북에서 진행해보고 결정해서 유틸로 분리하는 방식으로 진행


In [1]:
# 라이브러리 임포트
import re
from pathlib import Path

import pandas as pd
import numpy as np

# 경로 설정
DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")
REPORTS_DIR = Path("../reports")

# 출력 설정
pd.set_option("display.max_rows", None)  # 행 생략 없이 전부 표시
pd.set_option("display.max_columns", None)  # 열 생략 없이 전부 표시
pd.set_option("display.max_colwidth", None)  # 셀 안 텍스트 길이 안 자르고 전부 표시



In [2]:
# 데이터 로드
file_2021 = (
    DATA_DIR
    / "칼럼정렬"
    / "세부사업표추출_2021년도 지방자치단체 저출산고령사회 시행계획 (제4차 기본계획)_칼럼정렬.xlsx"
)
df_raw = pd.read_excel(
    file_2021,
    sheet_name="정리본_자동",
    usecols="C:N",
)

print(df_raw.shape)
df_raw.head(10)

(8115, 12)


,지역,세부사업명,사업분류재정구분,2021년 예산,2020년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,NaN,6921629,6745554,-188369,-6,NaN,1,4,6,데이터행
1,서울,1. 함께 일하고 함께 돌보는\n사회(공통),NaN,3178366,3366735,-1,-2,NaN,1,4,7,데이터행
2,서울,저출생 인식개선 캠페인,공통,43,44,-536,-3,"• 지역 저출산 극복 사회연대회의, 서울 100인의 아빠단\n운영, 저출산 극복 인식개선 캠페인등",1,4,8,데이터행
3,서울,어린이 안전 영상정보\n인프라 구축,공통,17569,18105,-3230,-86,• 어린이보호구역 내 과속단속카메라 등 설치,1,4,9,데이터행
4,서울,입양아동 가족지원,공통,520,3750,30636,11,"• 입양아동 양육수당, 장애아동 입양양육보조금 등",1,4,10,데이터행
5,서울,국공립어린이집 등\n보육서비스 수준제고,공통,321508,290872,-57664,-74,• 지원대상 : 국공립어린이집 등\n• 지원내용 : 보육교직원 인건비지원,1,4,11,데이터행
6,서울,국공립어린이집 확충,공통,20000,77664,2929,0,• 국공립어린이집 이용률 46%달성,1,4,12,데이터행
7,서울,어린이집 이용 영유아\n무상보육 확대,공통,883597,880668,-221,-11,• 지원대상 : 어린이집 이용 영유아\n• 지원내용 : 해당연령보육료,1,4,13,데이터행
8,서울,육아종합지원센터 운영,공통,1727,1948,-9818,-5,• 지원대상 : 서울시 영유아\n• 지원내용 : 어린이집보육 및 가정양육지원 사업 등\n지원,1,4,14,데이터행
9,서울,가정양육수당 지원,공통,174711,184529,-10,-4,• 지원대상 : 어린이집･유치원 미이용 0~86개월 미만 아동\n• 지원내용 : 10~20만원(월)양육수당지원,1,4,15,데이터행


In [4]:
# 기본검사
print("[데이터셋 크기]\n", df_raw.shape)
print("=====================================")
print("[데이터 타입]\n", df_raw.dtypes)
print("=====================================")
print("[각 컬럼별 결측치 개수 평균]\n", df_raw.isna().mean().round(3))
print("=====================================")
print("[중복 행 수]", df_raw.duplicated().sum())
print("=====================================")

# 지역(시도) 목록 확인
print("[지역(시도) 목록 확인]\n", df_raw["지역"].value_counts())

[데이터셋 크기]
 (8115, 12)
[데이터 타입]
 지역             str
세부사업명          str
사업분류재정구분       str
2021년 예산    object
2020년 예산    object
증감액         object
비율          object
주요내용           str
원본표구간        int64
머리글행         int64
원본행          int64
행유형            str
dtype: object
[각 컬럼별 결측치 개수 평균]
 지역          0.000
세부사업명       0.019
사업분류재정구분    0.040
2021년 예산    0.019
2020년 예산    0.022
증감액         0.025
비율          0.029
주요내용        0.023
원본표구간       0.000
머리글행        0.000
원본행         0.000
행유형         0.000
dtype: float64
[중복 행 수] 0
[지역(시도) 목록 확인]
 지역
경기    1256
경남     758
부산     750
충남     688
전남     575
전북     525
충북     491
강원     490
경북     484
인천     413
대전     329
대구     301
울산     283
서울     260
광주     222
제주     179
세종     111
Name: count, dtype: int64


In [5]:
# 2021년도 연속행 후보 검토표 생성
candidate_mask = df_raw["행유형"] != "데이터행"

review = df_raw.loc[
    candidate_mask,
    [
        "지역",
        "원본행",
        "행유형",
        "세부사업명",
        "사업분류재정구분",
        "2021년 예산",
        "2020년 예산",
        "주요내용",
    ],
].copy()

# 같은 지역 내 앞뒤 행의 핵심 정보 추가
grouped = df_raw.groupby("지역", sort=False)

review["앞행_세부사업명"] = grouped["세부사업명"].shift(1).loc[candidate_mask]
review["앞행_주요내용"] = grouped["주요내용"].shift(1).loc[candidate_mask]
review["뒷행_세부사업명"] = grouped["세부사업명"].shift(-1).loc[candidate_mask]
review["뒷행_주요내용"] = grouped["주요내용"].shift(-1).loc[candidate_mask]

review["판정"] = ""
review["판정근거"] = ""

review.to_csv(
    REPORTS_DIR / "2021_행유형_후보_검토.csv",
    index=False,
    encoding="utf-8-sig",
)

print(review["행유형"].value_counts())
print("총 후보:", len(review))

행유형
주요내용 연속행 후보     151
구분행사업명 연속 후보      6
Name: count, dtype: int64
총 후보: 157


In [6]:
# 후보 157건 판정 확정
review["판정"] = "제외"

review["판정근거"] = np.where(
    review["행유형"] == "주요내용 연속행 후보",
    "페이지 상단의 시도명 머리글로 확인되어 제외",
    "예산과 공통/자체 값이 없는 표 내부 구분행으로 확인되어 제외",
)

review.to_csv(
    REPORTS_DIR / "2021_행유형_후보_검토.csv",
    index=False,
    encoding="utf-8-sig",
)

print(review["판정"].value_counts())

display(
    review.loc[
        review["행유형"] == "구분행사업명 연속 후보",
        ["지역", "원본행", "세부사업명", "판정", "판정근거"],
    ]
)

판정
제외    157
Name: count, dtype: int64


,지역,원본행,세부사업명,판정,판정근거
131,서울,175,지원서비스 강화,제외,예산과 공통/자체 값이 없는 표 내부 구분행으로 확인되어 제외
362,부산,482,일반산업단지 조성,제외,예산과 공통/자체 값이 없는 표 내부 구분행으로 확인되어 제외
5717,전북,7454,행사지원,제외,예산과 공통/자체 값이 없는 표 내부 구분행으로 확인되어 제외
5864,전북,7643,운영지원),제외,예산과 공통/자체 값이 없는 표 내부 구분행으로 확인되어 제외
5958,전북,7760,운영,제외,예산과 공통/자체 값이 없는 표 내부 구분행으로 확인되어 제외
6103,전북,7949,추진,제외,예산과 공통/자체 값이 없는 표 내부 구분행으로 확인되어 제외


In [7]:
# 지역별로 데이터 분리
sido_dfs = {sido: group.copy() for sido, group in df_raw.groupby("지역")}

# 시도별 행 수 확인
for sido, df_sido in sido_dfs.items():
    print(sido, len(df_sido))

강원 490
경기 1256
경남 758
경북 484
광주 222
대구 301
대전 329
부산 750
서울 260
세종 111
울산 283
인천 413
전남 575
전북 525
제주 179
충남 688
충북 491


In [8]:
# 서울만 따로 확인 - #3 산출물 검증
df_seoul = sido_dfs["서울"]

print(df_seoul.shape)
df_seoul.head(10)

(260, 12)


,지역,세부사업명,사업분류재정구분,2021년 예산,2020년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형
0,서울,Ⅰ. 공통사업,NaN,6921629,6745554,-188369,-6,NaN,1,4,6,데이터행
1,서울,1. 함께 일하고 함께 돌보는\n사회(공통),NaN,3178366,3366735,-1,-2,NaN,1,4,7,데이터행
2,서울,저출생 인식개선 캠페인,공통,43,44,-536,-3,"• 지역 저출산 극복 사회연대회의, 서울 100인의 아빠단\n운영, 저출산 극복 인식개선 캠페인등",1,4,8,데이터행
3,서울,어린이 안전 영상정보\n인프라 구축,공통,17569,18105,-3230,-86,• 어린이보호구역 내 과속단속카메라 등 설치,1,4,9,데이터행
4,서울,입양아동 가족지원,공통,520,3750,30636,11,"• 입양아동 양육수당, 장애아동 입양양육보조금 등",1,4,10,데이터행
5,서울,국공립어린이집 등\n보육서비스 수준제고,공통,321508,290872,-57664,-74,• 지원대상 : 국공립어린이집 등\n• 지원내용 : 보육교직원 인건비지원,1,4,11,데이터행
6,서울,국공립어린이집 확충,공통,20000,77664,2929,0,• 국공립어린이집 이용률 46%달성,1,4,12,데이터행
7,서울,어린이집 이용 영유아\n무상보육 확대,공통,883597,880668,-221,-11,• 지원대상 : 어린이집 이용 영유아\n• 지원내용 : 해당연령보육료,1,4,13,데이터행
8,서울,육아종합지원센터 운영,공통,1727,1948,-9818,-5,• 지원대상 : 서울시 영유아\n• 지원내용 : 어린이집보육 및 가정양육지원 사업 등\n지원,1,4,14,데이터행
9,서울,가정양육수당 지원,공통,174711,184529,-10,-4,• 지원대상 : 어린이집･유치원 미이용 0~86개월 미만 아동\n• 지원내용 : 10~20만원(월)양육수당지원,1,4,15,데이터행


In [9]:
# 이전 데이터셋에서 확인한 classify_row 적용
sido_title_pattern = re.compile(r"붙\s*임\s*\(([^)]+)\)")


def classify_row(세부사업명: str) -> str:
    """대분류_소계 / 중분류_소계 / 헤더반복 / 세부사업 행 판별"""
    if pd.isna(세부사업명) or str(세부사업명).strip() == "":
        return "헤더반복"

    name = str(세부사업명).strip()

    if name == "세부사업명":
        return "헤더반복"
    if sido_title_pattern.search(name):
        return "헤더반복"
    if re.match(
        r"^[Ⅰ-Ⅿ]", name
    ):  # 유니코드 로마숫자 대문자 전체 블록(Ⅰ~ⅬⅭⅮⅯ) - 대분류 10개(Ⅹ) 초과 대비
        return "대분류_소계"
    if re.match(r"^\d+\.", name) and re.search(r"\((공통|자체)\)$", name):  # 조건 추가
        return "중분류_소계"
    return "세부사업"


df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)

# 예산 컬럼 숫자형 변환
for col in ["2021년 예산", "2020년 예산"]:
    df_raw[col] = pd.to_numeric(
        df_raw[col].astype(str).str.replace(",", "").replace("-", 0),
        errors="coerce",
    )

# 2021년 자료에 포함된 페이지 머리글·페이지 번호·구분행 제외
header_mask = (
    df_raw["2021년 예산"].isna()
    & df_raw["2020년 예산"].isna()
)

df_raw.loc[header_mask, "사업행구분"] = "헤더반복"

print(df_raw["사업행구분"].value_counts())

사업행구분
세부사업      7301
헤더반복       647
중분류_소계     133
대분류_소계      34
Name: count, dtype: int64


In [19]:
# 2021년 확인된 열 밀림 오류 1건 보정
fix_mask = (
    (df_raw["지역"] == "경북")
    & (df_raw["원본행"] == 8822)
)

assert fix_mask.sum() == 1

df_raw.loc[
    fix_mask,
    [
        "사업분류재정구분",
        "2021년 예산",
        "2020년 예산",
        "증감액",
        "비율",
    ],
] = [
    np.nan,
    147931.0,
    129831.0,
    18100.0,
    13.9,
]

display(
    df_raw.loc[
        fix_mask,
        [
            "지역",
            "원본행",
            "세부사업명",
            "사업분류재정구분",
            "2021년 예산",
            "2020년 예산",
            "증감액",
            "비율",
        ],
    ]
)

,지역,원본행,세부사업명,사업분류재정구분,2021년 예산,2020년 예산,증감액,비율
6789,경북,8822,4.인구구조 변화에 대한 적응(공통),NaN,147931.0,129831.0,18100.0,13.9


In [20]:
# df_raw["사업행구분"] = df_raw["세부사업명"].apply(classify_row)

# 시도별 행종류 분포
rowtype_distribution = df_raw.groupby(["지역", "사업행구분"]).size().unstack(fill_value=0)
rowtype_distribution

사업행구분,대분류_소계,세부사업,중분류_소계,헤더반복
지역,,,,
강원,2,437,8,43
경기,2,1149,8,97
경남,2,684,8,64
경북,2,439,6,37
광주,2,195,8,17
대구,2,267,8,24
대전,2,294,8,25
부산,2,685,8,55
서울,2,228,8,22


In [23]:
# 지역별로 원본 순서 유지한 채 대분류/중분류 라벨 세부사업행에 전파
def assign_labels(df_sido: pd.DataFrame) -> pd.DataFrame:
    """대분류_소계 / 중분류_소계 행의 이름을 뒤따르는 세부사업 행에 전파"""
    df_sido = df_sido.sort_values(
        "원본행"
    ).copy()  # 그냥 copy해도 되지만(raw도 이미 순서가 정렬되어있음) ffill이 순서에 의존하므로 원본 문서 순서를 명시적으로 보장함.
    major_mask = df_sido["사업행구분"] == "대분류_소계"
    medium_mask = df_sido["사업행구분"] == "중분류_소계"
    df_sido["대분류"] = df_sido["세부사업명"].where(major_mask).ffill()
    df_sido["중분류"] = df_sido["세부사업명"].where(medium_mask).ffill()

    return df_sido


df_labeled = df_raw.groupby("지역", group_keys=False).apply(assign_labels)
df_labeled["지역"] = df_raw.loc[df_labeled.index, "지역"]

# leaf(세부사업)만 추출
df_leaf = df_labeled[df_labeled["사업행구분"] == "세부사업"].copy()
print(df_leaf.shape)
print(df_leaf.columns.tolist())
df_leaf.head()

(7301, 15)
['세부사업명', '사업분류재정구분', '2021년 예산', '2020년 예산', '증감액', '비율', '주요내용', '원본표구간', '머리글행', '원본행', '행유형', '사업행구분', '대분류', '중분류', '지역']


,세부사업명,사업분류재정구분,2021년 예산,2020년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역
2,저출생 인식개선 캠페인,공통,43.0,44.0,-536,-3,"• 지역 저출산 극복 사회연대회의, 서울 100인의 아빠단\n운영, 저출산 극복 인식개선 캠페인등",1,4,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는\n사회(공통),서울
3,어린이 안전 영상정보\n인프라 구축,공통,17569.0,18105.0,-3230,-86,• 어린이보호구역 내 과속단속카메라 등 설치,1,4,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는\n사회(공통),서울
4,입양아동 가족지원,공통,520.0,3750.0,30636,11,"• 입양아동 양육수당, 장애아동 입양양육보조금 등",1,4,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는\n사회(공통),서울
5,국공립어린이집 등\n보육서비스 수준제고,공통,321508.0,290872.0,-57664,-74,• 지원대상 : 국공립어린이집 등\n• 지원내용 : 보육교직원 인건비지원,1,4,11,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는\n사회(공통),서울
6,국공립어린이집 확충,공통,20000.0,77664.0,2929,0,• 국공립어린이집 이용률 46%달성,1,4,12,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는\n사회(공통),서울


In [24]:
# 세부사업의 대분류·중분류 라벨 누락 검사
print("대분류 누락:", df_leaf["대분류"].isna().sum())
print("중분류 누락:", df_leaf["중분류"].isna().sum())

display(
    df_leaf.loc[
        df_leaf["대분류"].isna() | df_leaf["중분류"].isna(),
        [
            "지역",
            "원본행",
            "세부사업명",
            "대분류",
            "중분류",
        ],
    ]
)

대분류 누락: 0
중분류 누락: 0


,지역,원본행,세부사업명,대분류,중분류


In [25]:
mask_non_numeric = (
    pd.to_numeric(df_leaf["2021년 예산"], errors="coerce").isna() & df_leaf["2021년 예산"].notna()
)
display(df_leaf.loc[mask_non_numeric, ["지역", "세부사업명", "2021년 예산"]])

print("숫자로 변환되지 않은 예산 행:", mask_non_numeric.sum())

,지역,세부사업명,2021년 예산


숫자로 변환되지 않은 예산 행: 0


In [26]:
# QA: 중분류_소계 행 예산 vs leaf 합계 대조 (17개 시도 x 중분류 단위)
leaf_sum = (
    df_leaf.groupby(["지역", "중분류"])["2021년 예산"]
    .sum()
    .reset_index()
    .rename(columns={"2021년 예산": "leaf_합계"})
)

# 원본 소계 행 값 (중분류_소계 행에서 직접 가져온 값)
subtotal = df_labeled[df_labeled["사업행구분"] == "중분류_소계"][
    ["지역", "대분류", "세부사업명", "2021년 예산"]
].rename(columns={"세부사업명": "중분류", "2021년 예산": "원본_소계값"})

qa = subtotal.merge(leaf_sum, on=["지역", "중분류"], how="left")
qa["결과"] = np.where(qa["leaf_합계"] == qa["원본_소계값"], "일치", "불일치")
qa["결과"].value_counts()

결과
일치     113
불일치     20
Name: count, dtype: int64

In [27]:
# 중분류 소계 QA 불일치 21건 상세 확인
qa_mismatch = qa.loc[
    qa["결과"] == "불일치"
].copy()

# 이 줄이 빠져서 KeyError가 발생했음
qa_mismatch["차이"] = (
    qa_mismatch["leaf_합계"]
    - qa_mismatch["원본_소계값"]
)

qa_mismatch["절대차이"] = qa_mismatch["차이"].abs()

qa_mismatch = qa_mismatch.sort_values(
    ["절대차이", "지역"],
    ascending=[False, True],
)

display(
    qa_mismatch[
        [
            "지역",
            "대분류",
            "중분류",
            "원본_소계값",
            "leaf_합계",
            "차이",
            "절대차이",
        ]
    ]
)

print("불일치 건수:", len(qa_mismatch))

print("\n[절대차이 구간]")
print(
    pd.cut(
        qa_mismatch["절대차이"],
        bins=[-1, 0.01, 1, 10, 100, 1000, float("inf")],
        labels=[
            "0.01 이하",
            "0.01 초과–1",
            "1 초과–10",
            "10 초과–100",
            "100 초과–1,000",
            "1,000 초과",
        ],
    ).value_counts(sort=False)
)

,지역,대분류,중분류,원본_소계값,leaf_합계,차이,절대차이
116,경북,Ⅱ. 지자체 자체사업,2. 건강하고 능동적인\n고령사회 구축(자체),71660.9,68091.300,-3569.600,3569.600
86,충북,Ⅱ. 지자체 자체사업,3. 모두의 역량이 고루\n발휘되는 사회(자체),129817.7,129355.680,-462.020,462.020
30,인천,Ⅱ. 지자체 자체사업,3. 모두의 역량이 고루\n발휘되는 사회(자체),250902.0,250972.900,70.900,70.900
29,인천,Ⅱ. 지자체 자체사업,2. 건강하고 능동적인 고령사회 구축(자체),199501.8,199525.300,23.500,23.500
44,대전,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는\n사회(자체),70390.0,70394.000,4.000,4.000
91,충남,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는\n사회(자체),173210.0,173212.000,2.000,2.000
43,대전,Ⅰ. 공통사업,4.인구구조 변화에 대한\n적응(공통),28979.0,28980.000,1.000,1.000
46,대전,Ⅱ. 지자체 자체사업,3. 모두의 역량이 고루 발휘되는 사회(자체),71113.0,71114.000,1.000,1.000
47,대전,Ⅱ. 지자체 자체사업,4.인구구조 변화에 대한\n적응(자체),5834.0,5835.000,1.000,1.000
5,서울,Ⅱ. 지자체 자체사업,2. 건강하고 능동적인\n고령사회 구축(자체),169880.0,169881.000,1.000,1.000


불일치 건수: 20

[절대차이 구간]
절대차이
0.01 이하          0
0.01 초과–1       14
1 초과–10          2
10 초과–100        2
100 초과–1,000     1
1,000 초과         1
Name: count, dtype: int64


In [28]:
display(
    qa_mismatch.groupby("지역")
    .agg(
        불일치_건수=("차이", "size"),
        최대_절대차이=("절대차이", "max"),
        차이_합계=("차이", "sum"),
    )
    .sort_values(
        ["불일치_건수", "최대_절대차이"],
        ascending=False,
    )
)

,불일치_건수,최대_절대차이,차이_합계
지역,,,
대전,4,4.000,7.000
충남,3,2.000,4.000
경남,3,0.500,0.122
경북,2,3569.600,-3569.300
충북,2,462.020,-462.060
인천,2,70.900,94.400
서울,1,1.000,1.000
세종,1,1.000,-1.000
부산,1,0.500,-0.500


In [32]:
qa.head()

,지역,대분류,중분류,원본_소계값,leaf_합계,결과,차이
0,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는\n사회(공통),3178366.0,3178366.0,일치,0.0
1,서울,Ⅰ. 공통사업,2. 건강하고 능동적인\n고령사회(공통),3455119.0,3455119.0,일치,0.0
2,서울,Ⅰ. 공통사업,3. 모두의 역량이 고루 발휘되는 사회(공통),233636.0,233636.0,일치,0.0
3,서울,Ⅰ. 공통사업,4.인구구조 변화에 대한\n적응(공통),54508.0,54508.0,일치,0.0
4,서울,Ⅱ. 지자체 자체사업,1. 함께 일하고 함께 돌보는 사회(자체),257429.0,257429.0,일치,0.0


In [33]:
# QA 결과 확정 및 원인 분류
qa["차이"] = qa["leaf_합계"] - qa["원본_소계값"]
qa["절대차이"] = qa["차이"].abs()

qa["상대차이율"] = np.where(
    qa["원본_소계값"].abs() > 0,
    qa["절대차이"] / qa["원본_소계값"].abs() * 100,
    np.nan,
)

# 정확한 일치 여부
qa["결과"] = np.where(
    qa["절대차이"] <= 0.01,
    "일치",
    "불일치",
)

# 불일치 중 반올림 수준인지, 별도 확인이 필요한지 구분
qa["원인분류"] = np.select(
    [
        qa["절대차이"] <= 0.01,
        qa["상대차이율"] <= 0.1,
    ],
    [
        "일치",
        "미세차이(반올림·집계오차 가능)",
    ],
    default="원본 소계 불일치",
)

qa.to_csv(
    REPORTS_DIR / "2021_전국_QA_검증결과.csv",
    index=False,
    encoding="utf-8-sig",
)

print(qa["원인분류"].value_counts())
print("저장 완료:", REPORTS_DIR / "2021_전국_QA_검증결과.csv")

원인분류
일치                   113
미세차이(반올림·집계오차 가능)     18
원본 소계 불일치              2
Name: count, dtype: int64
저장 완료: ..\reports\2021_전국_QA_검증결과.csv


In [34]:
display(
    qa.loc[
        qa["원인분류"] == "원본 소계 불일치",
        [
            "지역",
            "대분류",
            "중분류",
            "원본_소계값",
            "leaf_합계",
            "차이",
            "상대차이율",
            "원인분류",
        ],
    ]
)

,지역,대분류,중분류,원본_소계값,leaf_합계,차이,상대차이율,원인분류
86,충북,Ⅱ. 지자체 자체사업,3. 모두의 역량이 고루\n발휘되는 사회(자체),129817.7,129355.68,-462.02,0.355899,원본 소계 불일치
116,경북,Ⅱ. 지자체 자체사업,2. 건강하고 능동적인\n고령사회 구축(자체),71660.9,68091.30,-3569.60,4.981238,원본 소계 불일치


# 증감액 / 비율 직접 재계산

- 위에서 확인했듯이 원본에서는 시도별로 부호나 값 자체가 틀린 경우가 있어 신뢰할 수 없음.
- 따라서 직접 재계산


In [35]:
def to_numeric_budget(series: pd.Series) -> pd.Series:
    """콤마 제거, '-'는 0으로 처리 후 숫자로 변환"""
    return pd.to_numeric(
        series.astype(str).str.replace(",", "").replace("-", 0),
        errors="coerce",
    )


df_leaf["2020년_예산_num"] = to_numeric_budget(df_leaf["2020년 예산"])
df_leaf["2021년_예산_num"] = to_numeric_budget(df_leaf["2021년 예산"])

df_leaf["증감액_재계산"] = (
    df_leaf["2021년_예산_num"]
    - df_leaf["2020년_예산_num"]
)

df_leaf["증감율_재계산"] = (
    df_leaf["증감액_재계산"]
    / df_leaf["2020년_예산_num"].replace(0, np.nan)
    * 100
).round(1)

df_leaf[
    [
        "세부사업명",
        "2021년_예산_num",
        "2020년_예산_num",
        "증감액_재계산",
        "증감율_재계산",
    ]
].head(10)

,세부사업명,2021년_예산_num,2020년_예산_num,증감액_재계산,증감율_재계산
2,저출생 인식개선 캠페인,43.0,44.0,-1.0,-2.3
3,어린이 안전 영상정보\n인프라 구축,17569.0,18105.0,-536.0,-3.0
4,입양아동 가족지원,520.0,3750.0,-3230.0,-86.1
5,국공립어린이집 등\n보육서비스 수준제고,321508.0,290872.0,30636.0,10.5
6,국공립어린이집 확충,20000.0,77664.0,-57664.0,-74.2
7,어린이집 이용 영유아\n무상보육 확대,883597.0,880668.0,2929.0,0.3
8,육아종합지원센터 운영,1727.0,1948.0,-221.0,-11.3
9,가정양육수당 지원,174711.0,184529.0,-9818.0,-5.3
10,부모 모니터링단 운영,272.0,282.0,-10.0,-3.5
11,공동육아나눔터,1776.0,1384.0,392.0,28.3


# 최종 스키마


In [36]:
# 텍스트 정리
PUA_PATTERN = re.compile(r"[\uE000-\uF8FF]")


def clean_text(series: pd.Series) -> pd.Series:
    """줄바꿈, 공백을 단일 공백으로 정리"""

    def _clean(x):
        if pd.isna(x):
            return x
        x = PUA_PATTERN.sub("", str(x))
        return re.sub(r"\s+", " ", x).strip()

    return series.apply(_clean)


df_leaf["세부사업명"] = clean_text(df_leaf["세부사업명"])
df_leaf["주요내용"] = clean_text(df_leaf["주요내용"])
df_leaf["대분류"] = clean_text(df_leaf["대분류"])
df_leaf["중분류"] = clean_text(df_leaf["중분류"])

df_leaf.head()

,세부사업명,사업분류재정구분,2021년 예산,2020년 예산,증감액,비율,주요내용,원본표구간,머리글행,원본행,행유형,사업행구분,대분류,중분류,지역,2020년_예산_num,2021년_예산_num,증감액_재계산,증감율_재계산
2,저출생 인식개선 캠페인,공통,43.0,44.0,-536,-3,"• 지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",1,4,8,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,44.0,43.0,-1.0,-2.3
3,어린이 안전 영상정보 인프라 구축,공통,17569.0,18105.0,-3230,-86,• 어린이보호구역 내 과속단속카메라 등 설치,1,4,9,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,18105.0,17569.0,-536.0,-3.0
4,입양아동 가족지원,공통,520.0,3750.0,30636,11,"• 입양아동 양육수당, 장애아동 입양양육보조금 등",1,4,10,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,3750.0,520.0,-3230.0,-86.1
5,국공립어린이집 등 보육서비스 수준제고,공통,321508.0,290872.0,-57664,-74,• 지원대상 : 국공립어린이집 등 • 지원내용 : 보육교직원 인건비지원,1,4,11,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,290872.0,321508.0,30636.0,10.5
6,국공립어린이집 확충,공통,20000.0,77664.0,2929,0,• 국공립어린이집 이용률 46%달성,1,4,12,데이터행,세부사업,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),서울,77664.0,20000.0,-57664.0,-74.2


In [37]:
# 컬럼명 정리 + 연도 추가 + 최종 스키마만 선택
df_leaf_final = (
    df_leaf.drop(columns=["증감액", "비율"])
    .rename(
        columns={
            "2021년_예산_num": "당해예산",
            "2020년_예산_num": "전년도예산",
            "증감액_재계산": "증감액",
            "증감율_재계산": "증감율",
        }
    )
    .assign(연도=2021)
)

df_leaf_final = df_leaf_final[
    [
        "연도",
        "지역",
        "대분류",
        "중분류",
        "사업분류재정구분",
        "세부사업명",
        "주요내용",
        "당해예산",
        "전년도예산",
        "증감액",
        "증감율",
        "원본행",
    ]
]

print(df_leaf_final.shape)
display(df_leaf_final.head())

(7301, 12)


,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,당해예산,전년도예산,증감액,증감율,원본행
2,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,저출생 인식개선 캠페인,"• 지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",43.0,44.0,-1.0,-2.3,8
3,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,어린이 안전 영상정보 인프라 구축,• 어린이보호구역 내 과속단속카메라 등 설치,17569.0,18105.0,-536.0,-3.0,9
4,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,입양아동 가족지원,"• 입양아동 양육수당, 장애아동 입양양육보조금 등",520.0,3750.0,-3230.0,-86.1,10
5,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립어린이집 등 보육서비스 수준제고,• 지원대상 : 국공립어린이집 등 • 지원내용 : 보육교직원 인건비지원,321508.0,290872.0,30636.0,10.5,11
6,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립어린이집 확충,• 국공립어린이집 이용률 46%달성,20000.0,77664.0,-57664.0,-74.2,12


In [38]:
year = 2021

for sido in df_leaf_final["지역"].unique():
    sido_leaf = df_leaf_final[df_leaf_final["지역"] == sido]
    sido_labeled = df_labeled[df_labeled["지역"] == sido]

    sido_dir = INTERIM_DIR / sido
    sido_dir.mkdir(parents=True, exist_ok=True)

    sido_leaf.to_csv(
        sido_dir / f"{year}_{sido}_세부사업_정제.csv", index=False, encoding="utf-8-sig"
    )
    sido_labeled.to_csv(
        sido_dir / f"{year}_{sido}_필터링전_전체원본.csv", index=False, encoding="utf-8-sig"
    )

# QA 결과는 전체 17개 시도 한 파일로 저장 (불일치 포함, 충남 결함도 그대로 남김)
qa.to_csv(REPORTS_DIR / f"{year}_전국_QA_검증결과.csv", index=False, encoding="utf-8-sig")

print("저장 완료: ", df_leaf_final["지역"].nunique(), "개 시도")

저장 완료:  17 개 시도


# 세부사업명 오탈자 / 중복 탐지

- 오탈자를 자동으로 고치지 않고, 사람이 검토할 후보만 찾는다.
- 같은 지역 / 중분류 안에서 완전히 같지는 않지만 유사도가 높은 세부사업명 쌍을 `rapidfuzz`로 찾는다.


In [40]:
from itertools import combinations
from rapidfuzz import fuzz

In [41]:
def normalize_for_match(name: str) -> str:
    """clean_text로 정제된 세부사업명에서, 비교 목적으로 공백, 문장부호 마저 제거"""
    return re.sub(r"[\s,-./\-/()]", "", name)


def find_near_duplicates(df: pd.DataFrame, threshold: int = 90) -> pd.DataFrame:
    """같은 지역 중분류 안에서 완전 일치는 아니지만 유사도가 높은 세부사업명과 쌍을 찾는다."""
    candidates = []
    for (sido, midium_cat), group in df.groupby(["지역", "중분류"]):
        rows = list(
            group[["세부사업명", "당해예산", "주요내용"]].itertuples(index=False, name=None)
        )
        for (name_a, budget_a, content_a), (name_b, budget_b, content_b) in combinations(rows, 2):
            if name_a == name_b:
                continue
            if normalize_for_match(name_a) == normalize_for_match(name_b):
                continue  # 공백/문장부호 차이 -> 검수 대상 x
            score = fuzz.ratio(name_a, name_b)
            if score >= threshold:
                candidates.append(
                    {
                        "지역": sido,
                        "중분류": midium_cat,
                        "세부사업명1": name_a,
                        "당해예산1": budget_a,
                        "주요내용1": content_a,
                        "세부사업명2": name_b,
                        "당해예산2": budget_b,
                        "주요내용2": content_b,
                        "유사도": score,
                    }
                )
    return pd.DataFrame(candidates).sort_values("유사도", ascending=False)


near_dup = find_near_duplicates(df_leaf_final)
print(len(near_dup), "건 발견")
display(near_dup.head(10))

221 건 발견


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
28,경기,1. 함께 일하고 함께 돌보는 사회(자체),"다자녀 가정, 한부모가정 수도요금 감면",0.0,• 지원내용 : 만18세 이하 자녀 세 명이상 동일세대 사용 요금 감면,"다자녀 가정, 한부모가정 하수도요금 감면",0.0,• 지원내용 : 만18세 이하 자녀 세 명이상 동일세대 사용 요금 감면,97.674419
53,경남,1. 함께 일하고 함께 돌보는 사회(자체),산모신생아 건강관리 본인부담금 지원,85.0,• 지원내용 : 산모신생아 건강관리 정부지원 바우처 대상자에게 정부지원금을 제외한 본인부담금 90%를 현금으로 지원(서비스 종료 후 환급),산모신생아 건강관리사 본인부담금 지원,15.0,• 지원내용 : 산모신생아 건강관리사(산후도우미) 본인부담금 지원,97.435897
76,부산,1. 함께 일하고 함께 돌보는 사회(자체),정부지원 어린이집 차량운영비 지원,168.0,• 통학버스를 운영하는 정부지원 어린이집에 차량운영비 지원(개소당 월50천원),정부미지원 어린이집 차량운영비 지원,72.0,• 부산광역시 사하구 소재 정부미지원 평가인증 어린이집에 차량운영비 지원,97.297297
75,부산,1. 함께 일하고 함께 돌보는 사회(자체),정부지원 어린이집 차량운영비 지원,168.0,• 통학버스를 운영하는 정부지원 어린이집에 차량운영비 지원(개소당 월50천원),정부미지원 어린이집 차량운영비 지원,6.6,"• 차량운영 중인 정부미지원(민간, 가정)어린이집 11개소 대상으로 차량 운영비(유류비) 연 지원",97.297297
74,부산,1. 함께 일하고 함께 돌보는 사회(자체),정부지원 어린이집 차량운영비 지원,168.0,• 통학버스를 운영하는 정부지원 어린이집에 차량운영비 지원(개소당 월50천원),정부미지원 어린이집 차량운영비 지원,9.0,• 정부미지원 어린이집 차량운영비 지원,97.297297
3,강원,2. 건강하고 능동적인 고령사회 구축(자체),저소득 노인세대 건강보험료 지원,77.0,• 대상 및 내용 : 저소득 노인가구 건강보험료 및 장기요양보험료 대납,저소득층 노인세대 건강보험료 지원,120.0,• 대상 및 내용 : 저소득 노인가구 건강보험료 및 장기요양보험료 대납,97.142857
59,경북,1. 함께 일하고 함께 돌보는 사회(자체),임산부 태아 기형아 검사비 지원,24.0,"• 지원대상 : 임신10주~20주 임산부 • 지원내용 : 회차당 쿠폰 35,000원지원",임부 태아 기형아 검사비 지원,24.0,• 지원내용 : 임부 태아 기형아검사비 지원,96.969697
57,경남,2. 건강하고 능동적인 고령사회 구축(자체),4대 이상 가정 효도수당 지원,20.0,• 지원내용 : 사천시에 3년 이상 주소지를 둔 4대 이상 가정에 설/명절 효도수당 각 50만원씩 지급,4세대 이상 가정 효도수당 지원,2.0,• 지원내용 : 군에 주소를 두고 4대 이상 함께 거주하는 가정에 매월 10만원 지원,96.969697
161,전남,1. 함께 일하고 함께 돌보는 사회(자체),산모신생아 건강관리 확대 지원,192.0,• 내용: 기준중위소득 초과자에 산모신생아 건강관리비 지원,산모･신생아 건강관리 확대 지원,23.0,• 내용: 정부지원 밖의 대상자에게 산모신생아 건강관리사 지원 (정부지원 대상자 제외),96.969697
38,경기,1. 함께 일하고 함께 돌보는 사회(자체),산모신생아 건강관리 지원사업,275.0,• 지원내용 : 소득기준 초과로 국비 지원 예외 출산 가정에 시비를 확보하여 건강관리사를 지원,산모･신생아 건강관리 지원사업,30.0,"• 지원대상 : 관내 등록 임산부 • 지원내용 : 산모･신생아 건강관리사 지원확대 (국고기준중위소득100%이하,자체-180%이하)",96.774194


In [42]:
# 당해예산이 완전히 같은 쌍은 진짜 같은 사업일 가능성이 높은 후보
near_dup_same_budget = near_dup[near_dup["당해예산1"] == near_dup["당해예산2"]]

print(len(near_dup_same_budget), "건 (전체", len(near_dup), "건 중)")
display(near_dup_same_budget)

7 건 (전체 221 건 중)


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
28,경기,1. 함께 일하고 함께 돌보는 사회(자체),"다자녀 가정, 한부모가정 수도요금 감면",0.0,• 지원내용 : 만18세 이하 자녀 세 명이상 동일세대 사용 요금 감면,"다자녀 가정, 한부모가정 하수도요금 감면",0.0,• 지원내용 : 만18세 이하 자녀 세 명이상 동일세대 사용 요금 감면,97.674419
59,경북,1. 함께 일하고 함께 돌보는 사회(자체),임산부 태아 기형아 검사비 지원,24.0,"• 지원대상 : 임신10주~20주 임산부 • 지원내용 : 회차당 쿠폰 35,000원지원",임부 태아 기형아 검사비 지원,24.0,• 지원내용 : 임부 태아 기형아검사비 지원,96.969697
37,경기,1. 함께 일하고 함께 돌보는 사회(자체),다자녀 가정 상수도 요금 감면,0.0,"• 지원내용 : 상수도요금 감면(가구당 매월 5톤/1,650원)",다자녀 가정 하수도 요금 감면,0.0,• 지원내용 : 다자녀 가정에 대한 하수도 요금 감면,93.750000
180,전북,1. 함께 일하고 함께 돌보는 사회(자체),다자녀 공무원 실적가점 부여,0.0,• 내용: 3자녀이상 공무원 실적가점 부여,다자녀 공무원 실적가점 부여사업,0.0,• 내용: 5급이하 공무원에 대해서 3자녀이상 출산공무원에 대 해 가점 부여,93.750000
51,경남,1. 함께 일하고 함께 돌보는 사회(자체),상수도요금 다자녀 감면,0.0,• 지원내용 : 주민등록표상 만19세미만 3자녀이상 세대에 10㎥에 해당하는 상수도요금 및 물이용 부담금 감면,하수도요금 다자녀 감면,0.0,• 지원내용 : 주민등록표상 만19세미만 3자녀이상 세대에 하수도요금 감면,91.666667
36,경기,1. 함께 일하고 함께 돌보는 사회(자체),다자녀 가정 종합사회복지관 수강료 감면,0.0,"• 지원대상 : 두자녀 이상, (막내자녀 만 15세이하) 가정 • 지원내용 : 가구당수강료 30%감면",다자녀 가정 미사강변종합사회복지관 수강료 감면,0.0,"• 지원대상 : 두자녀 이상, (막내자녀 만 15세이하) 가정 • 지원내용 : 가구당수강료30%감면",91.304348
50,경남,1. 함께 일하고 함께 돌보는 사회(자체),어린이날 행사 지원,30.0,• 지원내용 : 어린이날 행사 지원,어린이날 행사운영 지원,30.0,• 지원내용: 어린이날 행사 지원,90.909091


In [43]:
# 당해예산 0.0이 우연일치인지 확인
print("당해예산 = 0 으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] == 0).sum())
print("0이 아닌 값으로 일치한 건수: ", (near_dup_same_budget["당해예산1"] != 0).sum())

near_dup_confidenet = near_dup_same_budget[near_dup_same_budget["당해예산1"] != 0]
display(near_dup_confidenet)

당해예산 = 0 으로 일치한 건수:  5
0이 아닌 값으로 일치한 건수:  2


,지역,중분류,세부사업명1,당해예산1,주요내용1,세부사업명2,당해예산2,주요내용2,유사도
59,경북,1. 함께 일하고 함께 돌보는 사회(자체),임산부 태아 기형아 검사비 지원,24.0,"• 지원대상 : 임신10주~20주 임산부 • 지원내용 : 회차당 쿠폰 35,000원지원",임부 태아 기형아 검사비 지원,24.0,• 지원내용 : 임부 태아 기형아검사비 지원,96.969697
50,경남,1. 함께 일하고 함께 돌보는 사회(자체),어린이날 행사 지원,30.0,• 지원내용 : 어린이날 행사 지원,어린이날 행사운영 지원,30.0,• 지원내용: 어린이날 행사 지원,90.909091


# 주요내용 구조 패턴 검사

- 원자화를 위해 '지원대상:..', '지원내용: ..' 이 모든 행에 포함되어있는지 확인한다.


In [44]:
def dedup_label(text: str) -> str:
    """라벨 형식과 중복을 최소한으로 정리"""
    if pd.isna(text):
        return text

    text = re.sub(
        r"\(\s*(지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\s*\)",
        r"\1 : ",
        text,
    )

    # 2021년 자료의 '• 지원대상', '• 지원내용' 형식 처리
    text = re.sub(
        r"[•·]\s*(?=(지원대상|지원내용|사업대상|사업내용|대상|내용)\s*[:：])",
        "",
        text,
    )

    text = re.sub(r"(지원대상\s*[:：]\s*)+", "지원대상 : ", text)
    text = re.sub(r"(지원내용\s*[:：]\s*)+", "지원내용 : ", text)
    text = re.sub(r"(사업대상\s*[:：]\s*)+", "사업대상 : ", text)
    text = re.sub(r"(사업내용\s*[:：]\s*)+", "사업내용 : ", text)

    return text.strip()


df_leaf_final["주요내용"] = df_leaf_final["주요내용"].apply(dedup_label)

In [45]:
support_pattern = re.compile(r"^지원대상\s*[:：]\s*(.*?)\s*지원내용\s*[:：]\s*(.*)$", re.DOTALL)


def check_pattern(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern.match(text) else "불일치"


df_leaf_final["주요내용_패턴"] = df_leaf_final["주요내용"].apply(check_pattern)

print(df_leaf_final["주요내용_패턴"].value_counts())
print(df_leaf_final["주요내용_패턴"].value_counts(normalize=True).mul(100).round(1))

주요내용_패턴
불일치    6021
일치     1263
결측       17
Name: count, dtype: int64
주요내용_패턴
불일치    82.5
일치     17.3
결측      0.2
Name: proportion, dtype: float64


In [46]:
# 범위 넓혀보기
support_pattern_broad = re.compile(
    r"^(지원대상|사업대상|대상)\s*[:：]?\s*(.*?)\s*(지원내용|사업내용|내용)\s*[:：]?\s*(.*)$",
    re.DOTALL,
)


def check_pattern_broad(text: str) -> str:
    if pd.isna(text):
        return "결측"
    return "일치" if support_pattern_broad.match(text) else "불일치"


df_leaf_final["주요내용_패턴_확장"] = df_leaf_final["주요내용"].apply(check_pattern_broad)

print(df_leaf_final["주요내용_패턴_확장"].value_counts())
print(df_leaf_final["주요내용_패턴_확장"].value_counts(normalize=True).mul(100).round(1))

# 여전히 안걸리는 샘플 확인
display(
    df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"][
        ["지역", "세부사업명", "주요내용"]
    ].head(20)
)

주요내용_패턴_확장
불일치    5860
일치     1424
결측       17
Name: count, dtype: int64
주요내용_패턴_확장
불일치    80.3
일치     19.5
결측      0.2
Name: proportion, dtype: float64


,지역,세부사업명,주요내용
2,서울,저출생 인식개선 캠페인,"• 지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등"
3,서울,어린이 안전 영상정보 인프라 구축,• 어린이보호구역 내 과속단속카메라 등 설치
4,서울,입양아동 가족지원,"• 입양아동 양육수당, 장애아동 입양양육보조금 등"
6,서울,국공립어린이집 확충,• 국공립어린이집 이용률 46%달성
11,서울,공동육아나눔터,• 인건비 및 프로그램 등 운영비 지원
12,서울,아이돌봄서비스 확충 및 내실화,"• 인건비, 운영비 등"
13,서울,서울형 초등돌봄체계 우리동네키움센터 확충,• 우리동네 키움센터 설치
15,서울,아동청소년 자립지원 강화,"• 디딤씨앗통장, 드림스타트, 자립수당, 보호종료 아동주거지원 통합서비스"
17,서울,맞춤형 방과후학교운영,"• 방과후학교(돌봄) 지원센터 운영 • 방과후학교사업지원 : 초중고1,228교지원"
19,서울,학교 신설 및 재배치 검토,• 학교신설 토지 매입 및 공사


In [47]:
def extract_via_regex(text: str) -> dict:
    """패턴에 걸리면 그대로 분리, 안 걸리면 None"""
    if pd.isna(text):
        return {"지원대상": None, "지원내용": None}

    m = support_pattern_broad.match(text)

    if m:
        return {
            "지원대상": m.group(2).strip(),
            "지원내용": m.group(4).strip(),
        }

    return {"지원대상": None, "지원내용": None}


regex_extracted = pd.DataFrame(
    df_leaf_final["주요내용"]
    .apply(extract_via_regex)
    .tolist(),
    index=df_leaf_final.index,
)

df_leaf_final["지원대상"] = regex_extracted["지원대상"]
df_leaf_final["지원내용_상세"] = regex_extracted["지원내용"]

print(
    df_leaf_final[
        ["지원대상", "지원내용_상세"]
    ].notna().sum()
)

지원대상       1424
지원내용_상세    1424
dtype: int64


In [48]:
df_leaf_final[df_leaf_final["주요내용_패턴_확장"] == "불일치"].head()

,연도,지역,대분류,중분류,사업분류재정구분,세부사업명,주요내용,당해예산,전년도예산,증감액,증감율,원본행,주요내용_패턴,주요내용_패턴_확장,지원대상,지원내용_상세
2,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,저출생 인식개선 캠페인,"• 지역 저출산 극복 사회연대회의, 서울 100인의 아빠단 운영, 저출산 극복 인식개선 캠페인등",43.0,44.0,-1.0,-2.3,8,불일치,불일치,NaN,NaN
3,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,어린이 안전 영상정보 인프라 구축,• 어린이보호구역 내 과속단속카메라 등 설치,17569.0,18105.0,-536.0,-3.0,9,불일치,불일치,NaN,NaN
4,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,입양아동 가족지원,"• 입양아동 양육수당, 장애아동 입양양육보조금 등",520.0,3750.0,-3230.0,-86.1,10,불일치,불일치,NaN,NaN
6,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,국공립어린이집 확충,• 국공립어린이집 이용률 46%달성,20000.0,77664.0,-57664.0,-74.2,12,불일치,불일치,NaN,NaN
11,2021,서울,Ⅰ. 공통사업,1. 함께 일하고 함께 돌보는 사회(공통),공통,공동육아나눔터,• 인건비 및 프로그램 등 운영비 지원,1776.0,1384.0,392.0,28.3,17,불일치,불일치,NaN,NaN


# 주요내용 LLM 정제 (업스테이지 Solar Pro 3, 구조화 출력)

- LLM은 `주요내용_정제`(오탈자·이상한 공백만 보존형으로 교정) 생성만 담당한다. 요약/재구성 없음, 숫자·고유명사 변경 없음.
- `지원대상`/`지원내용_상세`는 이미 검증된 정규식(`extract_via_regex`) 결과를 그대로 쓴다 — 정규식으로 걸러지는 데 LLM을 또 쓰는 건 실익이 없고 과잉분리 위험만 늘린다.
- `response_format`(json_schema)으로 API 단에서 출력 구조를 강제, 실패 시 1회 재시도 후에도 실패하면 원문을 그대로 사용(정제문장 결측 방지).
- 정제 전후 숫자(금액·비율·연령 등) 보존 여부를 자동 검증해, 달라진 행은 검토 대상으로 표시한다.


In [50]:
import os
import json
import yaml
from openai import OpenAI
from tqdm import tqdm
from dotenv import load_dotenv

tqdm.pandas()

with open("../configs/llm_extraction.yaml", encoding="utf-8") as f:
    llm_cfg = yaml.safe_load(f)

load_dotenv("../.env")

api_key = os.getenv("UPSTAGE_API_KEY")

if not api_key:
    raise RuntimeError(
        "UPSTAGE_API_KEY가 없습니다. 프로젝트 루트의 .env 파일을 확인하세요."
    )

client = OpenAI(
    api_key=api_key,
    base_url=llm_cfg["upstage"]["base_url"],
)

RESPONSE_FORMAT = {"type": "json_schema", "json_schema": llm_cfg["response_schema"]}


def call_llm_once(name: str, content: str) -> str | None:
    """API 1회 호출. 파싱 실패하면 None 반환"""
    prompt = llm_cfg["prompt"]["template"].format(name=name, content=content)
    response = client.chat.completions.create(
        model=llm_cfg["upstage"]["model"],
        messages=[{"role": "user", "content": prompt}],
        temperature=llm_cfg["upstage"]["temperature"],
        response_format=RESPONSE_FORMAT,
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)["정제문장"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


def clean_sentence(name: str, content: str) -> str:
    """주요내용을 LLM으로 정제. 결측이면 결측 유지, 실패 시(1회 재시도 포함) 원문 그대로 사용"""
    if pd.isna(content):
        return None
    for attempt in range(2):  # 최초 시도 + 1회 재시도
        try:
            result = call_llm_once(name, content)
        except Exception as e:
            print(f"API 호출 실패({attempt + 1}회차): {name} -> {e}")
            result = None
        if result is not None:
            return result
    print(f"정제 실패, 원문 유지: {name}")
    return content


def extract_numbers(text) -> list:
    """숫자(금액·비율·연령 등) 시퀀스만 뽑아 리스트로 반환 - 원문/정제문장 보존 검증용"""
    if pd.isna(text):
        return []
    return re.findall(r"\d+", text)


def numbers_preserved(original, cleaned) -> bool:
    """정제 과정에서 숫자가 그대로 보존됐는지 확인 (다르면 검토 대상)"""
    return extract_numbers(original) == extract_numbers(cleaned)

# 먼저 소량 샘플로 속도·품질 확인 (전체 실행 전 확인용)

```python
sample = df_leaf_final.sample(
    llm_cfg["sample"]["size"], random_state=llm_cfg["sample"]["random_state"]
).copy()

sample["주요내용_정제"] = sample.progress_apply(
    lambda row: clean_sentence(row["세부사업명"], row["주요내용"]), axis=1
)
sample["숫자보존"] = sample.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~sample["숫자보존"]).sum())
display(
    sample[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세", "숫자보존"]]
)
```


샘플 결과가 괜찮으면 아래 셀로 전체 실행 (7천여 건, 시간 오래 걸릴 수 있음)


In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / "llm_정제_체크포인트.csv"

PUA_LOW, PUA_HIGH = chr(0xE000), chr(0xF8FF)
pua_re = re.compile(f"[{PUA_LOW}-{PUA_HIGH}]")
paren_label_re = re.compile(
    r"\((지원대상|지원내용|사업대상|사업내용|주요내용|전달체계|목적|대상|내용)\)"
)


def needs_llm_rerun(text):
    if pd.isna(text):
        return False
    return bool(pua_re.search(text)) or bool(paren_label_re.search(text))


# 체크포인트 파일을 직접 읽어서 판별한다(df_leaf_final 병합을 기다릴 필요 없음).
# 이렇게 해야 LLM 처리 셀보다 앞에 둘 수 있고, 한 번의 순차 실행(Run All)만으로
# 재실행 대상까지 자동으로 포함되어 처리된다 (CodeRabbit 리뷰 반영).
if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    affected_mask = checkpoint_df["주요내용_정제"].apply(needs_llm_rerun)
    affected_idx = checkpoint_df.index[affected_mask]

    if len(affected_idx) > 0:
        checkpoint_df = checkpoint_df.drop(index=affected_idx)
        checkpoint_df.to_csv(CHECKPOINT_PATH)

    print(
        "재실행 대상(체크포인트에서 제거):",
        len(affected_idx),
        "건 -> 남은 행수:",
        len(checkpoint_df),
    )
else:
    print("체크포인트 파일 없음 - 전체 신규 실행")

In [ ]:
CHECKPOINT_PATH = INTERIM_DIR / "llm_정제_체크포인트.csv"
CHUNK_SIZE = 200

if CHECKPOINT_PATH.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_PATH, index_col=0)
    done_index = set(checkpoint_df.index)
    print(f"체크포인트 발견: {len(done_index)}건 이미 처리됨, 이어서 진행")
else:
    checkpoint_df = pd.DataFrame(columns=["주요내용_정제"])
    done_index = set()

targets = df_leaf_final.index.difference(done_index)
results = {}

for i, idx in enumerate(tqdm(targets), start=1):
    row = df_leaf_final.loc[idx]
    results[idx] = clean_sentence(row["세부사업명"], row["주요내용"])

    if i % CHUNK_SIZE == 0:
        partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
        checkpoint_df = pd.concat([checkpoint_df, partial])
        checkpoint_df.to_csv(CHECKPOINT_PATH)
        results = {}

# 남은 것 마저 저장
if results:
    partial = pd.DataFrame.from_dict(results, orient="index", columns=["주요내용_정제"])
    checkpoint_df = pd.concat([checkpoint_df, partial])
    checkpoint_df.to_csv(CHECKPOINT_PATH)

df_leaf_final["주요내용_정제"] = checkpoint_df["주요내용_정제"]
df_leaf_final["숫자보존"] = df_leaf_final.apply(
    lambda row: numbers_preserved(row["주요내용"], row["주요내용_정제"]), axis=1
)

print("숫자 불일치(검토 대상) 건수:", (~df_leaf_final["숫자보존"]).sum())
display(
    df_leaf_final[~df_leaf_final["숫자보존"]][
        ["지역", "세부사업명", "주요내용", "주요내용_정제"]
    ].head(20)
)

df_leaf_final[["세부사업명", "주요내용", "주요내용_정제", "지원대상", "지원내용_상세"]].head(20)

In [ ]:
output_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "주요내용_정제",
    "당해예산",
    "전년도예산",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
]

missing_cols = [c for c in output_cols if c not in df_leaf_final.columns]

if missing_cols:
    raise KeyError(f"df_leaf_final에 없는 칼럼: {missing_cols}")

for sido_name, group in df_leaf_final.groupby("지역"):
    out_path = INTERIM_DIR / sido_name / f"2024_{sido_name}_세부사업_정제.csv"
    group[output_cols].to_csv(out_path, index=False)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")


print("\n숫자보존 불일치 총 건수:", (~df_leaf_final["숫자보존"]).sum())

In [ ]:
df_leaf_final.columns.tolist()

In [ ]:
id_cols = [
    "연도",
    "지역",
    "대분류",
    "중분류",
    "사업분류재정구분",
    "세부사업명",
    "주요내용",
    "증감액",
    "증감율",
    "원본행",
    "지원대상",
    "지원내용_상세",
    "주요내용_정제",
]

df_long = df_leaf_final.melt(
    id_vars=id_cols,
    value_vars=["당해예산", "전년도예산"],
    var_name="예산구분",
    value_name="예산액",
)

# 전년도예산 행은 실제 그 돈이 집행된 연도로 맞춰서 -1
previous_mask = df_long["예산구분"] == "전년도예산"
df_long.loc[previous_mask, "연도"] -= 1
# 증감액/증감율은 "당해 대비 전년" 개념이라 전년도 행에는 그대로 복제되면 안 됨 (CodeRabbit 지적)
df_long.loc[previous_mask, ["증감액", "증감율"]] = None

df_long = df_long.sort_values(["지역", "원본행", "예산구분"]).reset_index(drop=True)

print("long 변환 결과:", len(df_long), "행 (wide", len(df_leaf_final), "행 x 2)")
df_long.head(10)

In [ ]:
# 시도별로 저장
for sido_name, group in df_long.groupby("지역"):
    out_path = INTERIM_DIR / sido_name / f"2024_{sido_name}_세부사업_정제_long.csv"
    group.to_csv(out_path, index=False)
    print(f"{sido_name}: {len(group)}행 저장 -> {out_path}")